# GameTheory-04h (Rust) — Jeux de routage / congestion & Price of Anarchy

**Navigation** : [Version Python](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb) | [Version C#](GameTheory-04h-Routing-PriceOfAnarchy-CSharp.ipynb) | [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb) | [<< 4g-Supermodularité](GameTheory-04g-Supermodularite-Python.ipynb) | [Index](README.md)

**Objectifs** : calculer un équilibre de Wardrop et un optimum social sur Pigou, comparer leurs coûts, interpréter le PoA et traiter les cas où le point critique sort du domaine faisable.

**Prérequis** : fonctions affines, dérivée d'une fonction quadratique, équilibre de Nash — et, côté Rust, la lecture d'un `Result` et du opérateur `?`.

**Durée estimée** : 35 minutes.

Ce carnet est le **jumeau Rust** de [GameTheory-04h (Python)](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb) : même instance, mêmes formules, mêmes conclusions, mais écrites en **Rust standard** — `std` seul, aucune crate, aucune cellule `:dep`. Le kernel `rust` (evcxr) compile chaque cellule : chaque cellule est donc **autonome** (elle redéfinit tout ce qu'elle nomme) et le typage remplace le `raise` / `try` Python par `Result` et le propagateur `?`.

La différence de fond n'est pas cosmétique : là où le jumeau Python calcule en flottants et **admet** des entrées hors domaine (relancées par `raise ValueError`), le portage Rust **filtre** son domaine — chaque fonction de calcul renvoie un `Result`, et l'erreur se propage par `?`. Le §3bis remplace en outre `fractions.Fraction`, absent de `std`, par un **type rationnel local** écrit dans la cellule, qui reproduit l'arithmétique exacte.

**Références** : Tim Roughgarden, [*Selfish Routing and the Price of Anarchy*](https://mitpress.mit.edu/9780262182430/selfish-routing-and-the-price-of-anarchy/) (MIT Press, 2005) ; Tim Roughgarden et Éva Tardos, ["How Bad is Selfish Routing?"](https://www.timroughgarden.org/papers/routing.pdf), *Journal of the ACM* 49(2), 236–259, 2002. La borne 4/3 vaut sous les hypothèses précisées ci-dessous.

In [1]:
// Briques du modele : les deux fonctions de cout et le filtrage de domaine.
// Rust n'a pas d'argument par defaut : `alpha` et `d` se passent explicitement.
// `#[allow(dead_code)]` : une fonction definie et non appelee produit un warning
// sur stderr ; le carnet doit laisser stderr vide (discipline du lot).

#[allow(dead_code)]
fn cout_congestionne(x: f64, alpha: f64) -> f64 {
    // Cout par usager d'un lien congestionne : c(x) = alpha * x.
    alpha * x
}

#[allow(dead_code)]
fn cout_libre(_x: f64) -> f64 {
    // Cout par usager d'un lien 'libre' (cout constant, sans congestion).
    1.0
}

fn verifier_domaine(alpha: f64, d: f64) -> Result<(), String> {
    // Les quatre controles Python (math.isfinite sur alpha et D, puis > 0)
    // deviennent un Result : l'echec porte le message, l'appelant propage par `?`.
    if !(alpha.is_finite() && d.is_finite() && alpha > 0.0 && d > 0.0) {
        return Err("alpha et D doivent etre finis et strictement positifs".to_string());
    }
    Ok(())
}

#[allow(dead_code)]
fn optimum_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    // Optimum social avec x_A contraint a [0,D] et couts normalises.
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = d.min(0.5 / alpha);
    let x_b = d - x_a;
    let cout_total = d * (if t <= 0.5 { t } else { 1.0 - 0.25 / t });
    Ok((x_a, x_b, cout_total))
}

#[allow(dead_code)]
fn wardrop_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    // Equilibre Wardrop, flux conserves et cout social total.
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = if t <= 1.0 { d } else { d.min(1.0 / alpha) };
    let x_b = d - x_a;
    let cout_total = d * (if t <= 1.0 { t } else { 1.0 });
    Ok((x_a, x_b, cout_total))
}

#[allow(dead_code)]
fn poa_pigou(alpha: f64, d: f64) -> Result<f64, String> {
    // PoA stable en unites normalisees, y compris les limites de flottants.
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    Ok(if t <= 0.5 { 1.0 } else { t.min(1.0) / (1.0 - 0.25 / t) })
}

println!("Briques du modele chargees :");
println!("  lien congestionne : c(x) = alpha * x");
println!("  lien libre        : c(x) = 1");

Briques du modele chargees :


  lien congestionne : c(x) = alpha * x


  lien libre        : c(x) = 1


## 1. L'instance canonique : l'exemple de Pigou

Deux chemins en parallèle relient la source `s` à la destination `t`, avec une demande totale `D` d'usagers :

- **Autoroute** (lien congestionné) : le coût par usager est proportionnel au nombre d'usagers dessus, `c_A(x_A) = alpha * x_A`.
- **Route locale** (lien libre) : coût constant `c_B = 1`, sans congestion.

Chaque usager choisit un chemin. C'est l'exemple fondateur qui donne son nom au phénomène : la route locale à coût constant égal à 1 conduit les usagers égoïstes à se jeter tous sur l'autoroute, alors que l'optimum social les répartirait.

In [2]:
// OPTIMUM SOCIAL : min  alpha*x_A^2 + x_B   sous  x_A + x_B = D
//   (cout total = flux * cout par usager sur chaque lien)
//   condition du premier ordre : 2*alpha*x_A - 1 = 0  ->  x_A* = 1/(2*alpha)
// Rust n'a pas d'argument par defaut : D = 1.0 se passe explicitement.

fn verifier_domaine(alpha: f64, d: f64) -> Result<(), String> {
    if !(alpha.is_finite() && d.is_finite() && alpha > 0.0 && d > 0.0) {
        return Err("alpha et D doivent etre finis et strictement positifs".to_string());
    }
    Ok(())
}

fn optimum_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = d.min(0.5 / alpha);
    let x_b = d - x_a;
    let cout_total = d * (if t <= 0.5 { t } else { 1.0 - 0.25 / t });
    Ok((x_a, x_b, cout_total))
}

let alpha = 1.0;   // pente de congestion de l'autoroute
let d = 1.0;       // demande totale (normalisee)

let (xa_opt, xb_opt, cout_opt) = optimum_pigou(alpha, d).expect("domaine valide");

println!("Optimum social : x_A = {:.3},  x_B = {:.3}", xa_opt, xb_opt);
println!("Cout total a l'optimum  = {:.4}", cout_opt);

Optimum social : x_A = 0.500,  x_B = 0.500


Cout total a l'optimum  = 0.7500


## 2. L'équilibre de Wardrop (équilibre d'utilisateur / Nash)

À l'équilibre de Wardrop, **aucun usager ne peut réduire son coût personnel en changeant unilatéralement de chemin**. Tant que le coût de l'autoroute `c_A(x_A) = alpha * x_A` est inférieur au coût de la route locale (`1`), les usagers préfèrent l'autoroute : ils s'y massent jusqu'à ce que `alpha * x_A` atteigne `1`. Au-delà, plus personne n'y gagne à s'y ajouter.

Cet équilibre est un **équilibre de Nash** du jeu de routage : la stratégie de chaque joueur (son chemin) est une meilleure réponse à celle des autres.

In [3]:
// EQUILIBRE DE WARDROP : tous sur l'autoroute tant que c_A(D) = alpha*D <= 1
// (sinon l'autoroute coute plus que la route libre -> repartition jusqu'a c_A = 1)
// Cellule autonome : elle redetruit tout ce qu'elle nomme.

fn verifier_domaine(alpha: f64, d: f64) -> Result<(), String> {
    if !(alpha.is_finite() && d.is_finite() && alpha > 0.0 && d > 0.0) {
        return Err("alpha et D doivent etre finis et strictement positifs".to_string());
    }
    Ok(())
}

fn wardrop_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = if t <= 1.0 { d } else { d.min(1.0 / alpha) };
    let x_b = d - x_a;
    let cout_total = d * (if t <= 1.0 { t } else { 1.0 });
    Ok((x_a, x_b, cout_total))
}

let alpha = 1.0;
let d = 1.0;

let (xa_ue, xb_ue, cout_ue) = wardrop_pigou(alpha, d).expect("domaine valide");

println!("Equilibre de Wardrop : x_A = {:.3},  x_B = {:.3}", xa_ue, xb_ue);
println!("Cout total a l'equilibre = {:.4}", cout_ue);

Equilibre de Wardrop : x_A = 1.000,  x_B = 0.000


Cout total a l'equilibre = 1.0000


## 3. Le Price of Anarchy — mesuré

Le **price of anarchy** rapporte le coût à l'équilibre égoïste au coût à l'optimum social :

$$\text{PoA} = \frac{\text{cout total à l'équilibre de Wardrop}}{\text{cout total à l'optimum social}}$$

C'est le chiffre central du carnet : il **quantifie** le prix de l'intérêt individuel décentralisé. PoA = 1 signifie que l'équilibre est déjà optimal ; plus il est grand, plus l'égoïsme collectif est coûteux.

In [4]:
// Cellule autonome : elle redetruit optimum_pigou et wardrop_pigou, puis
// recalcule les deux couts avant de former le quotient.

fn verifier_domaine(alpha: f64, d: f64) -> Result<(), String> {
    if !(alpha.is_finite() && d.is_finite() && alpha > 0.0 && d > 0.0) {
        return Err("alpha et D doivent etre finis et strictement positifs".to_string());
    }
    Ok(())
}

fn optimum_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = d.min(0.5 / alpha);
    let x_b = d - x_a;
    let cout_total = d * (if t <= 0.5 { t } else { 1.0 - 0.25 / t });
    Ok((x_a, x_b, cout_total))
}

fn wardrop_pigou(alpha: f64, d: f64) -> Result<(f64, f64, f64), String> {
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    let x_a = if t <= 1.0 { d } else { d.min(1.0 / alpha) };
    let x_b = d - x_a;
    let cout_total = d * (if t <= 1.0 { t } else { 1.0 });
    Ok((x_a, x_b, cout_total))
}

// Rust affiche `true`/`false` ; la fonction locale rend la forme Python.
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let alpha = 1.0;
let d = 1.0;

let (_, _, cout_ue) = wardrop_pigou(alpha, d).expect("domaine valide");
let (_, _, cout_opt) = optimum_pigou(alpha, d).expect("domaine valide");

let poa = cout_ue / cout_opt;

println!("Price of Anarchy = {:.4} / {:.4} = {:.4}", cout_ue, cout_opt, poa);
println!("  -> l'egarement interesse coute {:.1} % de cout supplementaire", 100.0 * (poa - 1.0));
println!("  -> l'exemple de Pigou donne PoA = 4/3 = {:.4}  (coherence : {})", 4.0 / 3.0, py_bool((poa - 4.0 / 3.0).abs() < 1e-9));

Price of Anarchy = 1.0000 / 0.7500 = 1.3333


  -> l'egarement interesse coute 33.3 % de cout supplementaire


  -> l'exemple de Pigou donne PoA = 4/3 = 1.3333  (coherence : True)


## 3bis. Le PoA en forme exacte — calculable à la main

Le chiffre central du carnet est **mesuré** en flottants à la section 3 : `cout_ue / cout_opt = 1.0000 / 0.7500 = 1.3333`, puis confronté à un littéral `4/3` codé en dur. Ce témoin **calcule la forme exacte** en arithmétique rationnelle — ici le type local `Rationnel`, construit sur `i64` avec un PGCD d'Euclide, sans flottant et sans crate externe — à partir des données de l'instance canonique ($\alpha = 1$, $D = 1$), pour exhiber que le `1.3333` affiché **est précisément** la fraction $\frac{4}{3}$. Tout est **calculable à la main** : l'optimum social met $x_A^* = 1/(2\alpha) = 1/2$ (coût $3/4$), l'équilibre de Wardrop met tous les usagers sur l'autoroute ($\alpha D = 1 \le 1$, coût $1$), et le rapport des coûts totaux est $1 \div 3/4 = 4/3$.

> Les helpers `optimum_pigou` / `wardrop_pigou` du carnet renvoient des **flottants** ; la forme exacte n'est pas accessible par eux. Ce témoin recalcule la **même instance** en arithmétique rationnelle — ce n'est pas une réimplémentation du même calcul, c'est son changement de représentation (exact au lieu de flottant).

In [5]:
// Instance canonique (alpha = 1, D = 1) en arithmétique EXACTE.
// Rust n'a pas de type rationnel dans `std` : on en definit un local, sur i64,
// normalise par PGCD d'Euclide, signe porte par le numerateur.
fn pgcd(a: i64, b: i64) -> i64 {
    let mut x = a.abs();
    let mut y = b.abs();
    while y != 0 {
        let r = x % y;
        x = y;
        y = r;
    }
    x
}

#[derive(Clone, Copy)]
struct Rationnel { n: i64, d: i64 }

impl Rationnel {
    fn new(n: i64, d: i64) -> Rationnel {
        // Le denominateur nul n'est jamais atteint dans ce carnet (il vaut 1 ou 2) :
        // on rend la fraction nulle plutot que de paniquer.
        if d == 0 {
            return Rationnel { n: 0, d: 1 };
        }
        let (mut num, mut den) = (n, d);
        if den < 0 {
            num = -num;
            den = -den;
        }
        let g = pgcd(num, den);
        if g > 1 {
            num /= g;
            den /= g;
        }
        Rationnel { n: num, d: den }
    }
    fn add(&self, autre: &Rationnel) -> Rationnel {
        Rationnel::new(self.n * autre.d + autre.n * self.d, self.d * autre.d)
    }
    fn sub(&self, autre: &Rationnel) -> Rationnel {
        Rationnel::new(self.n * autre.d - autre.n * self.d, self.d * autre.d)
    }
    fn mul(&self, autre: &Rationnel) -> Rationnel {
        Rationnel::new(self.n * autre.n, self.d * autre.d)
    }
    fn div(&self, autre: &Rationnel) -> Rationnel {
        Rationnel::new(self.n * autre.d, self.d * autre.n)
    }
    fn carre(&self) -> Rationnel {
        Rationnel::new(self.n * self.n, self.d * self.d)
    }
    // Comparaison exacte a un entier (produit croise) : `alpha * D <= 1` reste exact.
    fn cmp_le_entier(&self, k: i64) -> bool {
        self.n * k <= self.d
    }
    fn vers_f64(&self) -> f64 {
        self.n as f64 / self.d as f64
    }
}

impl std::fmt::Display for Rationnel {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        if self.d == 1 {
            write!(f, "{}", self.n)
        } else {
            write!(f, "{}/{}", self.n, self.d)
        }
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let alpha = Rationnel::new(1, 1);
let d = Rationnel::new(1, 1);

// OPTIMUM SOCIAL :  x_A* = 1 / (2*alpha) ;  cout total = alpha*x_A*^2 + (D - x_A*)
let xa_opt = Rationnel::new(1, 1).div(&Rationnel::new(2, 1).mul(&alpha));
let xb_opt = d.sub(&xa_opt);
let cout_opt = alpha.mul(&xa_opt.carre()).add(&xb_opt);

// EQUILIBRE DE WARDROP : tous sur l'autoroute car alpha*D <= 1 ;  cout = alpha * x_A * D
let xa_ue = if alpha.mul(&d).cmp_le_entier(1) { d } else { Rationnel::new(1, 1).div(&alpha) };
let xb_ue = d.sub(&xa_ue);
let cout_ue = alpha.mul(&xa_ue).mul(&d);

// PRICE OF ANARCHY — FORME EXACTE
let poa_exact = cout_ue.div(&cout_opt);

println!("Optimum social EXACT : x_A = {},  x_B = {},  cout total = {}", xa_opt, xb_opt, cout_opt);
println!("Equilibre Wardrop    : x_A = {},  x_B = {},  cout total = {}", xa_ue, xb_ue, cout_ue);
println!("Price of Anarchy EXACT = {} / {} = {}", cout_ue, cout_opt, poa_exact);
println!("  -> le 1.3333 flottant de la section 3 EST la fraction {}", poa_exact);
println!("  -> controle de coherence avec le flottant mesure : {}",
    py_bool((poa_exact.vers_f64() - cout_ue.div(&cout_opt).vers_f64()).abs() < 1e-9));

Optimum social EXACT : x_A = 1/2,  x_B = 1/2,  cout total = 3/4


Equilibre Wardrop    : x_A = 1,  x_B = 0,  cout total = 1


Price of Anarchy EXACT = 1 / 3/4 = 4/3


  -> le 1.3333 flottant de la section 3 EST la fraction 4/3


  -> controle de coherence avec le flottant mesure : True


### Lecture du résultat (forme exacte)

Le `1.3333` mesuré en flottant (section 3) s'identifie à la fraction exacte **$\frac{4}{3}$** : l'optimum social coûte $\frac{3}{4}$, l'équilibre égoïste coûte $1$, et leur rapport est $1 \div \frac{3}{4} = \frac{4}{3}$. C'est **calculable à la main** en deux lignes. L'exemple de Pigou atteint donc **exactement** la borne $\frac{4}{3}$ de Roughgarden–Tardos annoncée à la section 4 — c'est son **cas d'acuité** : le PoA du réseau n'est pas seulement *borné* à $\frac{4}{3}$, il l'**atteint**. Le chiffre flottant et la borne théorique sont le même nombre, et la fraction $\frac{4}{3}$ en est la forme exacte.

## 4. La borne 4/3 (Roughgarden–Tardos, 2002)

**Théorème.** Pour un réseau à **une seule commodity** dont les coûts de lien sont **affines non décroissants** `c_e(x) = a_e x_e + b_e` avec `a_e >= 0` et `b_e >= 0` (modèle à flux fractionnaire), le price of anarchy est **borné** par 4/3 :

$$\text{PoA} \le 4/3$$

Cette borne est **serrée** : l'instance de Pigou mesurée ci-dessus la **atteint exactement** en `α = 1` (PoA = 4/3) — elle en est le **cas d'égalité**. Sous les hypothèses précédentes, la borne générale vaut 4/3. (Le « 5/2 » souvent cité relève des jeux de congestion **atomiques** — joueurs indivisibles — de Christodoulou–Koutsoupias, un modèle distinct de celui de ce carnet.)

> **Portée du compagnon Lean** : il formalise un résultat atomique distinct : un équilibre minimisant le potentiel de Rosenthal a un coût au plus deux fois l'optimum, et ses fixtures montrent aussi un équilibre de Nash coûtant `15/7` fois l'optimum. Il ne formalise pas ici la borne non atomique 4/3.

## 5. Sensibilité : le PoA en fonction de la pente de congestion

Faisons varier la pente `alpha` de l'autoroute (à demande `D = 1` fixe) et **mesurons** le PoA. On attend : plus la congestion est raide, plus l'écart se referme — le PoA **décroît** vers 1. Le maximum tombe en `alpha = 1`, où Pigou réalise l'égalité de la borne ; sur tout ce domaine le PoA mesuré doit rester **≤ 4/3**.

In [6]:
// Contrôles independants : ancrages usuels, frontieres, valeurs extremes et domaine.

fn verifier_domaine(alpha: f64, d: f64) -> Result<(), String> {
    if !(alpha.is_finite() && d.is_finite() && alpha > 0.0 && d > 0.0) {
        return Err("alpha et D doivent etre finis et strictement positifs".to_string());
    }
    Ok(())
}

fn poa_pigou(alpha: f64, d: f64) -> Result<f64, String> {
    verifier_domaine(alpha, d)?;
    let t = alpha * d;
    Ok(if t <= 0.5 { 1.0 } else { t.min(1.0) / (1.0 - 0.25 / t) })
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let fixtures: [(f64, f64, f64); 7] = [
    (1.0, 1.0, 4.0 / 3.0),
    (1.0, 0.1, 1.0),
    (0.25, 1.0, 1.0),
    (2.0, 2.0, 16.0 / 15.0),
    (1e-300, 1e-300, 1.0),
    (1e-300, 1e300, 4.0 / 3.0),
    (1e300, 1e300, 1.0),
];

let mut fixture_ok = true;
for &(a, d, attendu) in fixtures.iter() {
    match poa_pigou(a, d) {
        Ok(p) => {
            if !p.is_finite() || (p - attendu).abs() >= 1e-10 {
                fixture_ok = false;
            }
        }
        Err(_) => { fixture_ok = false; }
    }
}

let mut invalid_rejected = true;
for &(bad_a, bad_d) in [(0.0, 1.0), (1.0, f64::INFINITY), (f64::NAN, 1.0)].iter() {
    if poa_pigou(bad_a, bad_d).is_ok() {
        invalid_rejected = false;
    }
}

println!("Contrôles bornés : {} fixtures standard/extrêmes, PoA fini et oracle={}; paramètres invalides rejetés={}.",
    fixtures.len(), py_bool(fixture_ok), py_bool(invalid_rejected));

println!("{:>6} | {:>12} | {:>16}", "alpha", "PoA mesure", "<= 4/3 ?");
println!("{}", "-".repeat(34));
for a in [1.0f64, 1.2, 1.4, 1.5, 1.8, 2.0] {
    let p = poa_pigou(a, 1.0).expect("domaine valide");
    let statut = if p <= 4.0 / 3.0 + 1e-9 { "oui" } else { "non" };
    println!("{:>6.1} | {:>12.4} | {:>16}", a, p, statut);
};
// Le point-virgule final est requis sous evcxr : sans lui, la boucle est
// l'expression finale du bloc et evcxr publie un `execute_result` `()` que la
// source Python n'a pas.

Contrôles bornés : 7 fixtures standard/extrêmes, PoA fini et oracle=True; paramètres invalides rejetés=True.


 alpha |   PoA mesure |         <= 4/3 ?


----------------------------------


   1.0 |       1.3333 |              oui


   1.2 |       1.2632 |              oui


   1.4 |       1.2174 |              oui


   1.5 |       1.2000 |              oui


   1.8 |       1.1613 |              oui


   2.0 |       1.1429 |              oui


La mesure confirme le profil théorique : le PoA est **maximal en `α = 1`** — valeur 4/3, le cas d'égalité de la borne — et **décroît** vers 1 quand la pente de congestion croît. La raison est directe : à `D = 1`, dès que `α > 1` l'équilibre se répartit et le coût d'équilibre est **figé à 1**, tandis que l'optimum social (`1 − 1/(4α)`) s'en approche. Tout le domaine reste donc **≤ 4/3**, jamais au-dessus : c'est exactement la borne de Roughgarden–Tardos, et le 4/3 n'est pas un artefact du point de bord — c'est la borne elle-même, atteinte en `α = 1`. Le PoA n'est pas un mot d'esprit : c'est une **quantité mesurable** qui structure la conception des réseaux.

## 5bis. La borne 4/3 n'est pas universelle : elle dépend du **degré** du coût

La borne 4/3 du §4 est énoncée sous l'hypothèse de coûts de lien **affines**
`c_e(x) = a_e x_e + b_e`. Tout ce que ce carnet a **exécuté** jusqu'ici est
l'instance affine de Pigou (`c_A(x) = alpha * x`) : cette hypothèse n'a donc
jamais été mise à l'épreuve par le code — elle est annoncée en prose, pas
vérifiée. Ce paragraphe rend cette **limite** vérifiable.

On généralise l'instance de Pigou à un lien de congestion **polynomial**
`c_A(x) = x^d` (degré `d`, demande `D = 1`), on re-dérive l'optimum social et
l'équilibre de Wardrop, puis on mesure le PoA.

**Proposition** : la borne 4/3 **cesse de tenir** dès que le degré `d` dépasse 1.
**Critère d'affaiblissement (posé avant le calcul)** : si le PoA du Pigou
quadratique (`d = 2`) **excède 4/3**, alors « PoA ≤ 4/3 » ne décrit que le
régime **affine** — la limite est l'hypothèse de linéarité elle-même. S'il
restait ≤ 4/3, la borne serait plus générale que l'hypothèse affirmée.

In [7]:
// Le meme reseau de Pigou, mais l'autoroute a un cout polynomial c(x) = x^d (degre d).
// On re-derive l'optimum social et l'equilibre de Wardrop, puis on mesure le PoA.
//   optimum social :  min  x_A^(d+1) + x_B   s.c.  x_A + x_B = D
//                     ->  FOC  (d+1) x_A^d - 1 = 0  ->  x_A* = (1/(d+1))^(1/d)
//   equilibre Wardrop :  tous sur l'autoroute tant que D^d <= 1
//                        (a D = 1 :  D^d = 1  ->  cout total = D * D^d = 1)
// Rust n'a pas d'argument par defaut : D = 1.0 se passe explicitement.

fn poa_polynome(degre: i32, d: f64) -> Result<(f64, f64, f64), String> {
    if degre < 1 {
        return Err("degre doit etre >= 1".to_string());
    }
    let g = degre as f64;
    // optimum social (FOC : (d+1) x_A^d = 1, borne au flux faisable [0, D])
    let xa_opt = d.min((1.0 / (g + 1.0)).powf(1.0 / g));
    let cout_opt = xa_opt.powf(g + 1.0) + (d - xa_opt);
    // equilibre de Wardrop (D^d <= 1 : tous sur l'autoroute ; sinon x_A = 1)
    let (xa_ue, cout_ue) = if d.powf(g) <= 1.0 {
        (d, d * d.powf(g))
    } else {
        (1.0, d * 1.0)
    };
    Ok((cout_ue / cout_opt, xa_opt, cout_opt))
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// CONTROLE : le degre 1 (affine) doit reproduire EXACTEMENT le 4/3 du corps.
let (poa_d1, _, _) = poa_polynome(1, 1.0).expect("degre valide");
// CONTRE-EXEMPLE : le degre 2 (quadratique) depasse 4/3.
let (poa_d2, xa_d2, opt_d2) = poa_polynome(2, 1.0).expect("degre valide");

println!("degre 1 (affine, controle) : PoA = {:.4}  [4/3 = {:.4}]  coherent = {}",
    poa_d1, 4.0 / 3.0, py_bool((poa_d1 - 4.0 / 3.0).abs() < 1e-9));
println!("degre 2 (quadratique)      : PoA = {:.4}  [4/3 = {:.4}]  depasse 4/3 = {}",
    poa_d2, 4.0 / 3.0, py_bool(poa_d2 > 4.0 / 3.0 + 1e-9));
println!("   x_A* = {:.4}  (= 1/sqrt(3) = {:.4});  cout optimum = {:.4}  (= 1 - 2*sqrt(3)/9 = {:.4})",
    xa_d2, 1.0 / 3.0f64.sqrt(), opt_d2, 1.0 - 2.0 * 3.0f64.sqrt() / 9.0);
println!();
println!("{:>5} | {:>12} | {:>10}", "degre", "PoA mesure", "<= 4/3 ?");
println!("{}", "-".repeat(32));
for degre in [1i32, 2, 3, 4, 5] {
    let (p, _, _) = poa_polynome(degre, 1.0).expect("degre valide");
    let statut = if p <= 4.0 / 3.0 + 1e-9 { "oui" } else { "NON" };
    println!("{:>5} | {:>12.4} | {:>10}", degre, p, statut);
};
// Le point-virgule final est requis sous evcxr : sans lui, la boucle est
// l'expression finale du bloc et evcxr publie un `execute_result` `()` que la
// source Python n'a pas.

degre 1 (affine, controle) : PoA = 1.3333  [4/3 = 1.3333]  coherent = True


degre 2 (quadratique)      : PoA = 1.6258  [4/3 = 1.3333]  depasse 4/3 = True


   x_A* = 0.5774  (= 1/sqrt(3) = 0.5774);  cout optimum = 0.6151  (= 1 - 2*sqrt(3)/9 = 0.6151)


degre |   PoA mesure |   <= 4/3 ?


--------------------------------


    1 |       1.3333 |        oui


    2 |       1.6258 |        NON


    3 |       1.8956 |        NON


    4 |       2.1505 |        NON


    5 |       2.3944 |        NON


### Lecture du résultat

Le contrôle `degré 1` reproduit **exactement** le 4/3 du corps (33,3 %
d'égarement, comme au §3) — la généralisation est donc fidèle. Le
**contre-exemple** est net : dès le `degré 2` (coût quadratique), le PoA saute
à **1,63**, au-dessus de 4/3, et **croît** avec le degré (1,90 à `d = 3`, 2,15
à `d = 4`, 2,39 à `d = 5`).

La borne 4/3 du §4 n'est donc **pas** une propriété universelle des jeux de
routage : c'est une propriété de la **classe affine** (degré 1). La constante
4/3 est **bornée au degré** du coût de lien — l'hypothèse « coûts affines » du
§4 est **porteuse**, pas décorative. C'est précisément ce que rendait invisible
le fait que le carnet n'exécutait que l'instance affine : la **limite** de la
borne (ce qui la fait cesser de tenir) n'apparaît qu'en **sortant** de la
classe, et ici elle est **mesurée**, pas affirmée.

## 6. Exercices

### Exercice 1 — Équilibre de Wardrop général

Pour une instance à **2 liens en parallèle** (autoroute `c(x) = alpha*x`, route libre `c = 1`) et une demande `D`, calcule l'équilibre de Wardrop, c'est-à-dire la répartition `(x_A, x_B)` des usagers à l'équilibre.

**Indice** : à l'équilibre, un usager ne gagne rien à changer de lien — compare le coût `c_A(x_A)` de l'autoroute au coût `1` de la route locale.

**En Rust** : la fonction retourne `Option<(f64, f64)>` — `None` tant qu'elle n'est pas complétée.

In [8]:
/// Renvoie (x_A, x_B) a l'equilibre de Wardrop de l'instance a 2 liens.
///
/// Autoroute : c(x) = alpha*x.  Route libre : c = 1.  Demande totale : D.
#[allow(dead_code)]
fn wardrop_2liens(_alpha: f64, _d: f64) -> Option<(f64, f64)> {
    // TODO etudiant
    println!("Exercice à compléter");
    None // TODO: Some((x_A, x_B))
}

println!("Exercice 1 : fonction Wardrop stub conservee.");

Exercice 1 : fonction Wardrop stub conservee.


### Exercice 2 — La fonction PoA

Écris une fonction `poa(alpha, D)` qui renvoie le **price of anarchy mesuré** de l'instance de Pigou généralisée (autoroute `alpha*x`, route libre `1`, demande `D`). Elle doit reproduire la valeur `4/3` pour `alpha = 1, D = 1`.

**En Rust** : la fonction retourne `Option<f64>` — `None` tant qu'elle n'est pas complétée.

In [9]:
/// Renvoie le price of anarchy (cout Wardrop / cout optimum social).
///
/// Instance : autoroute c(x) = alpha*x, route libre c = 1, demande D.
/// Indice : recompose l'optimum social (FOC) et l'equilibre de Wardrop.
#[allow(dead_code)]
fn poa(_alpha: f64, _d: f64) -> Option<f64> {
    // TODO etudiant
    println!("Exercice à compléter");
    None // TODO: Some(poa)
}

println!("Exercice 2 : fonction PoA stub conservee.");

Exercice 2 : fonction PoA stub conservee.


### Exercice 3 — Le PoA en fonction de la demande

Pour l'instance canonique (`alpha = 1`), calcule le PoA pour une **série de valeurs de la demande** `D` sur `[1, 4]` et décris la tendance. Que devient le PoA quand la demande croît ? (Rappel : à l'équilibre, tous sont sur l'autoroute tant que `alpha * D <= 1` ; au-delà, le flux se répartit.)

**En Rust** : la fonction retourne `Option<f64>` ; Rust n'a pas d'argument par défaut, `alpha = 1.0` se passe explicitement.

In [10]:
/// Renvoie le PoA de l'instance canonique (alpha=1) pour une demande D.
#[allow(dead_code)]
fn poa_en_fonction_de_d(_d: f64, _alpha: f64) -> Option<f64> {
    // TODO etudiant
    println!("Exercice à compléter");
    None // TODO: Some(poa)
}

// (a completer) evaluer sur D = [1.0, 2.0, 3.0, 4.0] et afficher la tendance
println!("Exercice 3 : calcul de sensibilite a completer.");

Exercice 3 : calcul de sensibilite a completer.


## Conclusion

### À retenir

Le **price of anarchy** transforme une intuition (« les usagers égoïstes gâchent le réseau ») en **quantité mesurable** : sur l'exemple de Pigou, l'intérêt individuel décentralisé coûte **33 %** de supplément (PoA = 4/3) par rapport à l'optimum social, et la borne 4/3 de Roughgarden–Tardos, sous les hypothèses `a_e,b_e >= 0`, encadre ce prix — Pigou en est le cas d'égalité. Le compagnon Lean formalise un résultat atomique distinct : borne 2× pour un équilibre minimisant le potentiel de Rosenthal, avec témoin d'un équilibre coûtant 15/7 fois l'optimum ; il ne formalise pas la borne non atomique 4/3. C'est le pont entre la **théorie des jeux** (l'équilibre de Nash / Wardrop) et la **conception d'algorithmes et de réseaux** efficaces — la branche appliquée que ce carnet ajoute à la série.

**Côté Rust** : le carnet n'utilise que `std` — aucune dépendance externe (`:dep`). Le type rationnel de la section 3bis est un type local `Rationnel` sur `i64`, normalisé par PGCD d'Euclide, qui remplace `fractions.Fraction` ; les erreurs de domaine passent par `Result<_, String>` au lieu d'une exception. Le formatage des flottants impose la precision explicite (`{:.4}`) : Rust ecrit `1` la ou Python ecrit `1.0`.

## 8. Récapitulatif des exercices

1. [Équilibre de Wardrop général](#exercice-1--équilibre-de-wardrop-général) : flux conservés et conditions de coût sur les chemins utilisés.
2. [Fonction PoA](#exercice-2--la-fonction-poa) : quotient stable des coûts pour des paramètres intérieurs, de frontière et extrêmes.
3. [Variation de la demande](#exercice-3--le-poa-en-fonction-de-la-demande) : comparer les valeurs calculées pour `D = 1, 2, 3, 4`.

Les trois fonctions restent à compléter. Les valeurs de contrôle ne remplissent aucun stub.

## Vérifiez votre compréhension

1. Pourquoi faut-il borner l’optimum par D ?
2. Quel est le ratio canonique et quelle est la portée du compagnon Lean ?

<details>
<summary>Éléments de réponse</summary>

1. Un flux sur un chemin ne peut dépasser la demande totale ; si le point critique est au-delà, le minimum faisable se trouve en `x_A=D`.
2. Le ratio non atomique de Pigou vaut `1/0,75=4/3`. Le compagnon Lean formalise un jeu atomique et sa borne de potentiel 2×, et présente un témoin à 15/7 pour un autre équilibre de Nash.

</details>

## Pour aller plus loin

- [Jumeau Python](GameTheory-04h-Routing-PriceOfAnarchy-Python.ipynb)
- [Jumeau C#](GameTheory-04h-Routing-PriceOfAnarchy-CSharp.ipynb)
- [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)
- Roughgarden et Tardos (2002), *How Bad is Selfish Routing?*.